# IMD RF25 rainfall normals, 1991–2020, at mandi grid cells

**Purpose.** Compute standard 30-year (WMO/IMD 1991–2020) monthly rainfall normals from the IMD RF25 0.25° gridded daily rainfall, **at the exact grid cells where the onion mandis are located**. This keeps the baseline consistent with the 2021–2024 rainfall used in the paper, which is also measured at mandi locations.

**What this replaces.** The earlier `lpa_normals_2010_2020.csv` had three problems:
1. It was actually computed from **1990–1999** files (the notebook's own log), not 2010–2020.
2. It used **one hand-entered grid cell per subdivision** (some misplaced, e.g. "Coastal Andhra Pradesh" at 15.4°N 77.5°E, inland Rayalaseema).
3. It was compared with rainfall measured at mandi locations.

**Inputs.** The folder of yearly files `RF25_ind{YYYY}_rfp25.nc`. 1991–2020 is required; 2021–2024 is optional but recommended.

**Output.** `rf25_mandi_cells_1991_2020.xlsx` (send this back), with three sheets:
- `metadata`: run details and checks
- `cell_month_normals`: one row per grid cell × month, with mean, SD and number of years (1991–2020)
- `cell_year_month`: one row per grid cell × year × month, with the monthly total and valid days (all years processed)

The same tables are also written as CSV files.

Run time is about 2–5 minutes. The only required packages are `netCDF4`, `numpy`, `pandas` and `openpyxl`, which come with Anaconda.

In [ ]:
# !pip install netCDF4 openpyxl     # uncomment on first run if needed
import os, time, datetime
import numpy as np
import pandas as pd
import netCDF4 as nc
print('netCDF4', nc.__version__, '| pandas', pd.__version__)

## 1. Configuration

In [ ]:
NC_DIR = r"C:\Users\hp\OneDrive\Attachments\Desktop\netcdf 1990 2020"   # folder with RF25_ind{YYYY}_rfp25.nc
NORMAL_YEARS = list(range(1991, 2021))      # WMO / IMD standard 30-year normal period
EXTRA_YEARS  = list(range(2021, 2025))      # processed if the files exist (optional)
FNAME = 'RF25_ind{y}_rfp25.nc'
OUT_XLSX = 'rf25_mandi_cells_1991_2020.xlsx'

present = {y: os.path.exists(os.path.join(NC_DIR, FNAME.format(y=y))) for y in NORMAL_YEARS + EXTRA_YEARS}
missing_normal = [y for y in NORMAL_YEARS if not present[y]]
print('Normal-period files found:', sum(present[y] for y in NORMAL_YEARS), 'of 30')
print('Missing normal-period years:', missing_normal or 'none')
print('Optional 2021-2024 files found:', [y for y in EXTRA_YEARS if present[y]] or 'none')
assert not missing_normal, 'All 30 years 1991-2020 are needed for a standard normal.'

## 2. Grid cells needed

Centres of the 0.25° RF25 cells containing every verified mandi location (all 1,546 mandis; in-state coordinate variants within 50 km of the verified location). Generated by `00_geocode_check.py` in the replication package. Do not edit.

In [ ]:
CELLS = [
    (8.00, 77.50),
    (8.50, 77.00),
    (8.50, 77.50),
    (8.75, 78.00),
    (9.00, 76.50),
    (9.00, 76.75),
    (9.00, 77.25),
    (9.00, 77.50),
    (9.25, 77.00),
    (9.50, 76.50),
    (9.50, 77.75),
    (9.50, 78.00),
    (9.50, 78.50),
    (9.75, 76.50),
    (9.75, 76.75),
    (9.75, 77.00),
    (9.75, 77.25),
    (9.75, 77.50),
    (9.75, 78.50),
    (10.00, 76.25),
    (10.00, 76.50),
    (10.00, 78.00),
    (10.00, 78.75),
    (10.25, 78.75),
    (10.50, 76.25),
    (10.50, 77.75),
    (10.50, 79.75),
    (10.75, 76.50),
    (10.75, 76.75),
    (10.75, 77.00),
    (10.75, 77.50),
    (10.75, 78.50),
    (10.75, 78.75),
    (10.75, 79.25),
    (10.75, 79.50),
    (11.00, 76.00),
    (11.00, 77.00),
    (11.00, 79.50),
    (11.00, 79.75),
    (11.25, 75.75),
    (11.25, 78.00),
    (11.25, 79.00),
    (11.25, 79.25),
    (11.25, 79.75),
    (11.50, 76.00),
    (11.50, 76.75),
    (11.50, 77.25),
    (11.50, 78.25),
    (11.50, 79.25),
    (11.75, 78.25),
    (11.75, 79.00),
    (11.75, 79.75),
    (12.00, 75.25),
    (12.00, 75.50),
    (12.00, 77.00),
    (12.00, 79.50),
    (12.25, 76.25),
    (12.25, 76.50),
    (12.25, 76.75),
    (12.25, 78.00),
    (12.25, 79.00),
    (12.50, 75.00),
    (12.50, 75.25),
    (12.50, 75.75),
    (12.50, 78.00),
    (12.50, 78.50),
    (12.50, 79.00),
    (12.50, 79.50),
    (12.75, 74.75),
    (12.75, 75.25),
    (12.75, 76.75),
    (12.75, 77.25),
    (12.75, 78.50),
    (12.75, 79.25),
    (12.75, 79.50),
    (12.75, 80.00),
    (13.00, 74.75),
    (13.00, 76.25),
    (13.00, 77.50),
    (13.00, 79.00),
    (13.00, 79.50),
    (13.00, 79.75),
    (13.00, 80.00),
    (13.00, 80.25),
    (13.25, 75.75),
    (13.25, 77.00),
    (13.25, 77.25),
    (13.25, 77.50),
    (13.25, 78.25),
    (13.25, 80.00),
    (13.50, 74.75),
    (13.50, 75.75),
    (13.50, 77.00),
    (13.50, 77.75),
    (13.75, 75.75),
    (14.00, 75.25),
    (14.00, 75.50),
    (14.25, 75.75),
    (14.25, 76.50),
    (14.50, 76.00),
    (14.75, 75.50),
    (15.25, 75.25),
    (15.25, 76.50),
    (15.25, 76.75),
    (15.50, 73.75),
    (15.50, 74.00),
    (15.50, 75.75),
    (15.50, 76.25),
    (15.50, 77.50),
    (16.00, 74.50),
    (16.25, 75.00),
    (16.25, 75.50),
    (16.25, 75.75),
    (16.25, 77.25),
    (16.50, 74.00),
    (16.50, 78.25),
    (16.75, 74.25),
    (16.75, 76.00),
    (16.75, 78.00),
    (16.75, 79.25),
    (16.75, 79.50),
    (17.25, 73.50),
    (17.25, 74.25),
    (17.25, 74.50),
    (17.25, 76.75),
    (17.25, 77.00),
    (17.25, 77.75),
    (17.25, 78.50),
    (17.25, 80.50),
    (17.50, 76.00),
    (17.50, 78.00),
    (17.50, 78.50),
    (17.75, 74.25),
    (17.75, 75.25),
    (17.75, 75.50),
    (17.75, 76.00),
    (17.75, 77.75),
    (17.75, 79.75),
    (18.00, 74.25),
    (18.00, 74.50),
    (18.00, 77.25),
    (18.00, 78.25),
    (18.00, 78.75),
    (18.00, 79.00),
    (18.00, 79.50),
    (18.25, 73.25),
    (18.25, 74.50),
    (18.25, 75.75),
    (18.25, 76.00),
    (18.25, 76.75),
    (18.50, 73.25),
    (18.50, 73.75),
    (18.75, 73.00),
    (18.75, 73.75),
    (18.75, 74.00),
    (18.75, 74.75),
    (18.75, 78.00),
    (18.75, 78.25),
    (19.00, 72.75),
    (19.00, 73.25),
    (19.00, 74.00),
    (19.00, 74.50),
    (19.00, 74.75),
    (19.00, 76.00),
    (19.00, 79.50),
    (19.25, 73.25),
    (19.25, 74.00),
    (19.25, 74.75),
    (19.25, 79.50),
    (19.25, 84.25),
    (19.50, 74.50),
    (19.50, 74.75),
    (19.50, 75.00),
    (19.50, 78.50),
    (19.50, 81.00),
    (19.50, 83.50),
    (19.50, 84.75),
    (19.75, 74.25),
    (19.75, 75.25),
    (19.75, 78.50),
    (19.75, 83.00),
    (19.75, 84.25),
    (19.75, 85.25),
    (19.75, 85.75),
    (20.00, 73.75),
    (20.00, 74.00),
    (20.00, 74.25),
    (20.00, 74.50),
    (20.00, 75.25),
    (20.00, 75.75),
    (20.00, 79.50),
    (20.00, 83.25),
    (20.25, 73.75),
    (20.25, 74.00),
    (20.25, 74.25),
    (20.25, 74.50),
    (20.25, 74.75),
    (20.25, 82.75),
    (20.25, 83.25),
    (20.25, 85.00),
    (20.25, 85.75),
    (20.50, 73.75),
    (20.50, 74.25),
    (20.50, 75.00),
    (20.50, 76.50),
    (20.50, 82.50),
    (20.50, 83.50),
    (20.50, 84.75),
    (20.50, 85.75),
    (20.50, 86.00),
    (20.50, 86.50),
    (20.75, 73.00),
    (20.75, 75.50),
    (20.75, 76.50),
    (20.75, 76.75),
    (20.75, 77.25),
    (20.75, 78.50),
    (20.75, 83.25),
    (20.75, 83.75),
    (20.75, 84.00),
    (20.75, 85.25),
    (20.75, 85.50),
    (20.75, 86.00),
    (20.75, 86.75),
    (21.00, 70.75),
    (21.00, 73.00),
    (21.00, 74.25),
    (21.00, 75.00),
    (21.00, 75.75),
    (21.00, 77.75),
    (21.00, 79.75),
    (21.00, 80.75),
    (21.00, 85.00),
    (21.00, 85.50),
    (21.00, 86.50),
    (21.25, 70.25),
    (21.25, 70.50),
    (21.25, 70.75),
    (21.25, 72.75),
    (21.25, 73.00),
    (21.25, 74.50),
    (21.25, 75.75),
    (21.25, 76.25),
    (21.25, 77.75),
    (21.25, 79.00),
    (21.25, 79.25),
    (21.25, 81.50),
    (21.25, 83.50),
    (21.50, 69.75),
    (21.50, 70.50),
    (21.50, 72.00),
    (21.50, 74.50),
    (21.50, 79.25),
    (21.50, 83.75),
    (21.50, 84.25),
    (21.50, 85.50),
    (21.50, 86.50),
    (21.50, 86.75),
    (21.50, 87.00),
    (21.75, 70.50),
    (21.75, 72.25),
    (21.75, 73.00),
    (21.75, 75.00),
    (21.75, 76.25),
    (21.75, 78.25),
    (21.75, 84.00),
    (21.75, 84.25),
    (21.75, 85.50),
    (21.75, 86.75),
    (21.75, 87.25),
    (22.00, 71.00),
    (22.00, 75.75),
    (22.00, 76.50),
    (22.00, 78.00),
    (22.00, 84.00),
    (22.00, 85.00),
    (22.00, 85.75),
    (22.00, 86.50),
    (22.00, 86.75),
    (22.00, 87.50),
    (22.00, 87.75),
    (22.25, 70.25),
    (22.25, 70.75),
    (22.25, 73.00),
    (22.25, 73.25),
    (22.25, 74.25),
    (22.25, 74.75),
    (22.25, 77.00),
    (22.25, 78.75),
    (22.25, 79.75),
    (22.25, 82.25),
    (22.25, 83.25),
    (22.25, 87.50),
    (22.25, 88.00),
    (22.25, 88.25),
    (22.25, 88.50),
    (22.50, 70.00),
    (22.50, 72.75),
    (22.50, 73.00),
    (22.50, 74.50),
    (22.50, 75.25),
    (22.50, 75.75),
    (22.50, 77.25),
    (22.50, 77.75),
    (22.50, 78.00),
    (22.50, 87.25),
    (22.50, 88.00),
    (22.50, 88.25),
    (22.75, 69.75),
    (22.75, 71.00),
    (22.75, 71.50),
    (22.75, 71.75),
    (22.75, 72.25),
    (22.75, 73.00),
    (22.75, 75.75),
    (22.75, 76.00),
    (22.75, 76.25),
    (22.75, 77.75),
    (22.75, 87.75),
    (22.75, 88.00),
    (22.75, 88.25),
    (22.75, 88.50),
    (22.75, 88.75),
    (23.00, 72.50),
    (23.00, 73.00),
    (23.00, 74.25),
    (23.00, 74.75),
    (23.00, 75.25),
    (23.00, 77.00),
    (23.00, 79.25),
    (23.00, 81.75),
    (23.00, 86.25),
    (23.00, 87.25),
    (23.00, 88.00),
    (23.00, 88.50),
    (23.25, 75.75),
    (23.25, 76.75),
    (23.25, 77.50),
    (23.25, 78.25),
    (23.25, 80.00),
    (23.25, 86.25),
    (23.25, 86.50),
    (23.25, 87.25),
    (23.25, 87.75),
    (23.25, 88.25),
    (23.25, 88.50),
    (23.25, 91.75),
    (23.50, 70.00),
    (23.50, 72.25),
    (23.50, 72.50),
    (23.50, 75.00),
    (23.50, 76.50),
    (23.50, 76.75),
    (23.50, 77.50),
    (23.50, 84.75),
    (23.50, 87.25),
    (23.50, 88.00),
    (23.50, 88.50),
    (23.50, 91.25),
    (23.50, 91.50),
    (23.75, 73.75),
    (23.75, 75.25),
    (23.75, 75.50),
    (23.75, 76.75),
    (23.75, 77.00),
    (23.75, 78.75),
    (23.75, 79.50),
    (23.75, 80.25),
    (23.75, 87.00),
    (23.75, 87.75),
    (23.75, 88.25),
    (23.75, 91.25),
    (23.75, 91.50),
    (23.75, 91.75),
    (23.75, 92.00),
    (24.00, 74.50),
    (24.00, 74.75),
    (24.00, 75.75),
    (24.00, 76.25),
    (24.00, 78.25),
    (24.00, 83.00),
    (24.00, 83.75),
    (24.00, 87.75),
    (24.00, 88.75),
    (24.00, 91.50),
    (24.00, 91.75),
    (24.00, 92.25),
    (24.25, 72.25),
    (24.25, 72.50),
    (24.25, 75.50),
    (24.25, 76.50),
    (24.25, 87.75),
    (24.25, 88.25),
    (24.25, 92.00),
    (24.50, 72.75),
    (24.50, 73.50),
    (24.50, 73.75),
    (24.50, 74.50),
    (24.50, 74.75),
    (24.50, 77.25),
    (24.50, 83.00),
    (24.50, 88.00),
    (24.50, 92.25),
    (24.50, 93.75),
    (24.50, 94.00),
    (24.75, 71.75),
    (24.75, 72.75),
    (24.75, 74.50),
    (24.75, 75.25),
    (24.75, 78.50),
    (24.75, 79.75),
    (24.75, 80.75),
    (24.75, 81.50),
    (24.75, 83.00),
    (24.75, 84.50),
    (24.75, 84.75),
    (24.75, 85.50),
    (24.75, 86.25),
    (24.75, 86.75),
    (24.75, 92.50),
    (24.75, 93.00),
    (24.75, 93.75),
    (24.75, 94.00),
    (25.00, 75.25),
    (25.00, 76.75),
    (25.00, 82.75),
    (25.00, 83.50),
    (25.00, 84.00),
    (25.00, 88.25),
    (25.25, 72.25),
    (25.25, 73.25),
    (25.25, 73.75),
    (25.25, 75.75),
    (25.25, 76.00),
    (25.25, 77.75),
    (25.25, 79.75),
    (25.25, 80.50),
    (25.25, 81.00),
    (25.25, 81.75),
    (25.25, 82.00),
    (25.25, 82.50),
    (25.25, 83.00),
    (25.25, 83.25),
    (25.25, 84.75),
    (25.25, 85.00),
    (25.25, 85.50),
    (25.25, 85.75),
    (25.25, 86.25),
    (25.25, 86.50),
    (25.25, 87.25),
    (25.25, 88.00),
    (25.25, 88.25),
    (25.25, 90.50),
    (25.50, 73.50),
    (25.50, 74.75),
    (25.50, 76.75),
    (25.50, 78.75),
    (25.50, 79.25),
    (25.50, 79.50),
    (25.50, 80.50),
    (25.50, 80.75),
    (25.50, 81.25),
    (25.50, 81.50),
    (25.50, 81.75),
    (25.50, 82.50),
    (25.50, 83.50),
    (25.50, 84.00),
    (25.50, 84.50),
    (25.50, 85.00),
    (25.50, 85.25),
    (25.50, 86.00),
    (25.50, 86.75),
    (25.50, 87.75),
    (25.50, 88.50),
    (25.50, 90.25),
    (25.50, 90.75),
    (25.50, 91.00),
    (25.50, 92.25),
    (25.50, 93.50),
    (25.75, 72.25),
    (25.75, 79.00),
    (25.75, 80.00),
    (25.75, 81.00),
    (25.75, 81.75),
    (25.75, 82.25),
    (25.75, 82.50),
    (25.75, 84.00),
    (25.75, 84.75),
    (25.75, 85.50),
    (25.75, 85.75),
    (25.75, 86.50),
    (25.75, 87.00),
    (25.75, 87.50),
    (25.75, 88.00),
    (26.00, 73.75),
    (26.00, 75.25),
    (26.00, 75.75),
    (26.00, 78.25),
    (26.00, 79.00),
    (26.00, 79.50),
    (26.00, 79.75),
    (26.00, 80.25),
    (26.00, 80.50),
    (26.00, 82.75),
    (26.00, 83.00),
    (26.00, 83.25),
    (26.00, 83.50),
    (26.00, 84.00),
    (26.00, 84.75),
    (26.00, 86.00),
    (26.00, 88.00),
    (26.00, 89.50),
    (26.00, 93.75),
    (26.25, 73.00),
    (26.25, 74.75),
    (26.25, 76.25),
    (26.25, 78.25),
    (26.25, 79.75),
    (26.25, 81.25),
    (26.25, 81.75),
    (26.25, 82.25),
    (26.25, 82.50),
    (26.25, 83.75),
    (26.25, 84.50),
    (26.25, 85.25),
    (26.25, 86.25),
    (26.25, 86.75),
    (26.25, 87.25),
    (26.25, 88.00),
    (26.25, 89.50),
    (26.25, 90.00),
    (26.25, 91.00),
    (26.25, 91.75),
    (26.25, 92.75),
    (26.25, 93.75),
    (26.50, 73.00),
    (26.50, 74.75),
    (26.50, 77.75),
    (26.50, 79.50),
    (26.50, 79.75),
    (26.50, 80.00),
    (26.50, 80.25),
    (26.50, 80.50),
    (26.50, 82.50),
    (26.50, 83.75),
    (26.50, 84.25),
    (26.50, 85.25),
    (26.50, 85.50),
    (26.50, 87.00),
    (26.50, 89.00),
    (26.50, 89.25),
    (26.50, 89.50),
    (26.50, 90.00),
    (26.50, 94.50),
    (26.75, 78.25),
    (26.75, 79.00),
    (26.75, 79.25),
    (26.75, 79.50),
    (26.75, 80.75),
    (26.75, 81.00),
    (26.75, 81.75),
    (26.75, 82.00),
    (26.75, 82.25),
    (26.75, 82.50),
    (26.75, 83.00),
    (26.75, 83.25),
    (26.75, 83.50),
    (26.75, 83.75),
    (26.75, 84.00),
    (26.75, 84.25),
    (26.75, 85.00),
    (26.75, 88.25),
    (26.75, 88.50),
    (26.75, 88.75),
    (26.75, 89.50),
    (26.75, 92.75),
    (26.75, 94.25),
    (27.00, 70.75),
    (27.00, 73.75),
    (27.00, 75.75),
    (27.00, 77.25),
    (27.00, 78.00),
    (27.00, 78.50),
    (27.00, 79.25),
    (27.00, 79.75),
    (27.00, 80.00),
    (27.00, 80.25),
    (27.00, 81.25),
    (27.00, 81.50),
    (27.00, 82.00),
    (27.00, 83.25),
    (27.00, 83.75),
    (27.00, 84.25),
    (27.00, 88.25),
    (27.00, 88.50),
    (27.00, 95.00),
    (27.25, 75.75),
    (27.25, 77.75),
    (27.25, 78.00),
    (27.25, 78.25),
    (27.25, 78.50),
    (27.25, 79.00),
    (27.25, 80.00),
    (27.25, 80.50),
    (27.25, 82.75),
    (27.25, 83.00),
    (27.25, 83.50),
    (27.50, 76.50),
    (27.50, 77.75),
    (27.50, 78.00),
    (27.50, 78.50),
    (27.50, 79.25),
    (27.50, 79.50),
    (27.50, 80.75),
    (27.50, 81.75),
    (27.50, 82.00),
    (27.50, 82.50),
    (27.50, 83.50),
    (27.50, 95.00),
    (27.50, 95.50),
    (27.75, 74.50),
    (27.75, 75.00),
    (27.75, 76.25),
    (27.75, 77.00),
    (27.75, 77.50),
    (27.75, 78.25),
    (27.75, 78.75),
    (27.75, 80.75),
    (27.75, 81.50),
    (28.00, 73.00),
    (28.00, 73.25),
    (28.00, 75.50),
    (28.00, 76.00),
    (28.00, 77.00),
    (28.00, 77.25),
    (28.00, 77.75),
    (28.00, 78.25),
    (28.00, 79.00),
    (28.00, 79.75),
    (28.00, 80.50),
    (28.00, 80.75),
    (28.25, 74.75),
    (28.25, 75.50),
    (28.25, 76.25),
    (28.25, 76.50),
    (28.25, 77.25),
    (28.25, 77.50),
    (28.25, 77.75),
    (28.25, 79.00),
    (28.25, 80.75),
    (28.50, 76.00),
    (28.50, 76.50),
    (28.50, 76.75),
    (28.50, 77.00),
    (28.50, 77.25),
    (28.50, 77.50),
    (28.50, 77.75),
    (28.50, 78.00),
    (28.50, 78.50),
    (28.50, 79.00),
    (28.50, 79.50),
    (28.50, 79.75),
    (28.50, 80.00),
    (28.50, 80.25),
    (28.50, 80.50),
    (28.75, 76.00),
    (28.75, 76.25),
    (28.75, 77.00),
    (28.75, 77.25),
    (28.75, 77.50),
    (28.75, 77.75),
    (28.75, 78.00),
    (28.75, 78.25),
    (28.75, 78.50),
    (28.75, 79.25),
    (28.75, 79.50),
    (29.00, 75.25),
    (29.00, 76.50),
    (29.00, 77.00),
    (29.00, 77.25),
    (29.00, 77.75),
    (29.00, 78.00),
    (29.00, 78.25),
    (29.00, 78.50),
    (29.00, 78.75),
    (29.00, 79.50),
    (29.00, 79.75),
    (29.25, 73.50),
    (29.25, 74.00),
    (29.25, 74.25),
    (29.25, 74.50),
    (29.25, 74.75),
    (29.25, 75.75),
    (29.25, 76.00),
    (29.25, 76.25),
    (29.25, 77.00),
    (29.25, 77.25),
    (29.25, 77.75),
    (29.25, 78.75),
    (29.25, 79.50),
    (29.25, 80.00),
    (29.50, 73.50),
    (29.50, 74.00),
    (29.50, 74.25),
    (29.50, 74.75),
    (29.50, 75.00),
    (29.50, 75.50),
    (29.50, 76.25),
    (29.50, 76.75),
    (29.50, 77.00),
    (29.50, 77.25),
    (29.50, 77.50),
    (29.50, 77.75),
    (29.50, 78.25),
    (29.50, 78.50),
    (29.50, 79.50),
    (29.75, 73.50),
    (29.75, 74.00),
    (29.75, 75.00),
    (29.75, 75.25),
    (29.75, 75.50),
    (29.75, 76.50),
    (29.75, 77.00),
    (29.75, 77.25),
    (29.75, 77.75),
    (29.75, 78.00),
    (29.75, 78.75),
    (30.00, 74.00),
    (30.00, 75.00),
    (30.00, 75.25),
    (30.00, 75.50),
    (30.00, 76.00),
    (30.00, 76.75),
    (30.00, 77.25),
    (30.00, 77.50),
    (30.00, 77.75),
    (30.00, 78.00),
    (30.00, 78.25),
    (30.25, 74.00),
    (30.25, 74.25),
    (30.25, 74.50),
    (30.25, 74.75),
    (30.25, 75.00),
    (30.25, 75.50),
    (30.25, 75.75),
    (30.25, 76.00),
    (30.25, 76.25),
    (30.25, 77.00),
    (30.25, 77.25),
    (30.25, 78.00),
    (30.25, 78.75),
    (30.50, 74.25),
    (30.50, 74.50),
    (30.50, 74.75),
    (30.50, 75.00),
    (30.50, 75.25),
    (30.50, 75.75),
    (30.50, 76.00),
    (30.50, 76.75),
    (30.50, 77.50),
    (30.50, 77.75),
    (30.75, 74.50),
    (30.75, 75.00),
    (30.75, 75.25),
    (30.75, 75.75),
    (30.75, 76.00),
    (30.75, 76.25),
    (30.75, 76.50),
    (30.75, 76.75),
    (30.75, 77.00),
    (30.75, 77.50),
    (31.00, 74.50),
    (31.00, 74.75),
    (31.00, 75.00),
    (31.00, 75.25),
    (31.00, 75.75),
    (31.00, 76.00),
    (31.00, 76.25),
    (31.00, 76.50),
    (31.00, 76.75),
    (31.00, 77.00),
    (31.00, 77.25),
    (31.25, 74.75),
    (31.25, 75.50),
    (31.25, 75.75),
    (31.25, 76.00),
    (31.25, 76.25),
    (31.25, 76.50),
    (31.25, 76.75),
    (31.50, 74.75),
    (31.50, 75.00),
    (31.50, 75.25),
    (31.50, 75.75),
    (31.50, 76.25),
    (31.75, 74.75),
    (31.75, 75.00),
    (31.75, 75.75),
    (31.75, 76.50),
    (31.75, 77.00),
    (32.00, 75.25),
    (32.00, 75.50),
    (32.00, 76.25),
    (32.00, 77.00),
    (32.00, 77.50),
    (32.25, 75.75),
    (32.50, 75.50),
    (32.75, 74.75),
    (32.75, 75.00),
    (32.75, 76.25),
    (33.00, 74.75),
    (33.00, 75.25),
    (33.25, 74.25),
    (33.25, 74.75),
    (33.25, 75.25),
    (33.50, 74.25),
    (33.50, 74.75),
    (33.75, 75.00),
    (33.75, 75.25),
    (34.00, 74.25),
    (34.00, 74.75),
    (34.00, 75.00),
    (34.25, 74.25),
]
print(len(CELLS), 'grid cells')

## 3. Snap cells to the grid and check the land mask

If a cell is masked (no data: coast or border), the nearest valid cell within two grid steps is used and this is recorded.

In [ ]:
ds0 = nc.Dataset(os.path.join(NC_DIR, FNAME.format(y=NORMAL_YEARS[0])))
LAT = np.asarray(ds0.variables['LATITUDE'][:], dtype=float)
LON = np.asarray(ds0.variables['LONGITUDE'][:], dtype=float)
rf0 = ds0.variables['RAINFALL'][:]
fill = getattr(ds0.variables['RAINFALL'], '_FillValue', -999.0)
rf0 = np.ma.filled(rf0.astype('float32'), np.nan); rf0[rf0 == fill] = np.nan; rf0[rf0 < 0] = np.nan
valid = ~np.all(np.isnan(rf0), axis=0)          # land mask: cell has any data in the year
ds0.close()
print('Grid:', LAT.min(), LAT.max(), LON.min(), LON.max(), '| shape', len(LAT), 'x', len(LON), '| valid cells', int(valid.sum()))

rows = []
for clat, clon in CELLS:
    i = int(np.argmin(np.abs(LAT - clat))); j = int(np.argmin(np.abs(LON - clon)))
    ui, uj, moved = i, j, 0.0
    if not valid[i, j]:
        best = None
        for di in range(-2, 3):
            for dj in range(-2, 3):
                a, b = i + di, j + dj
                if 0 <= a < len(LAT) and 0 <= b < len(LON) and valid[a, b]:
                    d = np.hypot(di, dj)
                    if best is None or d < best[0]:
                        best = (d, a, b)
        if best is not None:
            _, ui, uj = best; moved = best[0] * 0.25
        else:
            ui = uj = -1
    rows.append({'cell_lat': clat, 'cell_lon': clon, 'grid_i': ui, 'grid_j': uj,
                 'used_lat': LAT[ui] if ui >= 0 else np.nan, 'used_lon': LON[uj] if uj >= 0 else np.nan,
                 'moved_deg': moved, 'no_valid_cell': ui < 0})
CELLMAP = pd.DataFrame(rows)
print('Cells moved to a neighbouring valid cell:', int((CELLMAP.moved_deg > 0).sum()))
print('Cells with no valid data nearby:', int(CELLMAP.no_valid_cell.sum()))

## 4. Extract daily rainfall and aggregate to monthly totals

In [ ]:
ok = CELLMAP[~CELLMAP.no_valid_cell].reset_index(drop=True)
gi, gj = ok.grid_i.values.astype(int), ok.grid_j.values.astype(int)
records = []
t0 = time.time()
for y in [y for y in NORMAL_YEARS + EXTRA_YEARS if present[y]]:
    ds = nc.Dataset(os.path.join(NC_DIR, FNAME.format(y=y)))
    assert np.allclose(ds.variables['LATITUDE'][:], LAT) and np.allclose(ds.variables['LONGITUDE'][:], LON), f'grid differs in {y}'
    t = ds.variables['TIME']
    dates = nc.num2date(t[:], t.units, only_use_cftime_datetimes=False, only_use_python_datetimes=True)
    months = np.array([d.month for d in dates]); years = np.array([d.year for d in dates])
    rf = np.ma.filled(ds.variables['RAINFALL'][:].astype('float32'), np.nan)[:, gi, gj]
    ds.close()
    rf[(rf == fill) | (rf < 0)] = np.nan
    assert (years == y).all(), f'file {y} contains dates from other years'
    for m in range(1, 13):
        block = rf[months == m]                      # days x cells
        tot = np.nansum(block, axis=0)
        nval = np.sum(~np.isnan(block), axis=0)
        ndays = block.shape[0]
        tot = np.where(nval > 0, tot * ndays / np.maximum(nval, 1), np.nan)   # scale up if a few days missing
        records.append(pd.DataFrame({'cell_lat': ok.cell_lat, 'cell_lon': ok.cell_lon, 'year': y, 'month': m,
                                     'rain_mm': tot, 'valid_days': nval, 'days_in_month': ndays}))
    print(f'{y}: done ({time.time()-t0:.0f}s)')
CYM = pd.concat(records, ignore_index=True)
print(CYM.shape); CYM.head()

## 5. 1991–2020 normals (mean and SD of monthly totals across the 30 years)

In [ ]:
base = CYM[CYM.year.isin(NORMAL_YEARS)]
NORM = (base.groupby(['cell_lat', 'cell_lon', 'month'])['rain_mm']
        .agg(normal_mean_mm='mean', normal_sd_mm='std', n_years='count').reset_index())
print('n_years per cell-month:', NORM.n_years.min(), '-', NORM.n_years.max())
print('Share of cell-months with SD = 0 (always dry):', round((NORM.normal_sd_mm == 0).mean(), 4))
# Sanity: mean June-September total across cells (should be several hundred mm)
jjas = NORM[NORM.month.isin([6, 7, 8, 9])].groupby(['cell_lat', 'cell_lon']).normal_mean_mm.sum()
print('June-Sept normal across cells: median', round(jjas.median(), 1), 'mm; range', round(jjas.min(), 1), '-', round(jjas.max(), 1))
NORM.head()

## 6. Save (send the .xlsx back)

In [ ]:
meta = pd.DataFrame({'item': ['source', 'normal_period', 'years_processed', 'n_cells_requested', 'n_cells_with_data',
                              'n_cells_moved_to_neighbour', 'grid', 'run_time_utc', 'netCDF4_version'],
                     'value': ['IMD RF25 gridded daily rainfall (0.25 deg)', '1991-2020',
                               ', '.join(str(y) for y in sorted(CYM.year.unique())), len(CELLS), len(ok),
                               int((CELLMAP.moved_deg > 0).sum()),
                               f'{LAT.min()}-{LAT.max()}N, {LON.min()}-{LON.max()}E, {len(LAT)}x{len(LON)}',
                               datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), nc.__version__]})
with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as xw:
    meta.to_excel(xw, sheet_name='metadata', index=False)
    NORM.to_excel(xw, sheet_name='cell_month_normals', index=False)
    CELLMAP.to_excel(xw, sheet_name='cell_map', index=False)
    CYM.to_excel(xw, sheet_name='cell_year_month', index=False)
NORM.to_csv('rf25_cell_month_normals_1991_2020.csv', index=False)
CYM.to_csv('rf25_cell_year_month.csv', index=False)
print('Saved', os.path.abspath(OUT_XLSX))
print('Rows: normals', len(NORM), '| cell-year-month', len(CYM))